In [2]:
# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
import os
import copy
import numpy as np
import pandas as pd
import open3d as o3d

EXPERIMENT = "test_9_simulation_3"
WORKPIECE = "TH0011AV"
SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{WORKPIECE}"
OUT_DIR = f"simulation/{EXPERIMENT}/{WORKPIECE}"


# EXPERIMENT = "test_6_noise"
# SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}"
# OUT_DIR = f"simulation/{EXPERIMENT}"

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

TARGET_VIEWPOINT = 17

# ------------------------------------------
# NOISE PARAMETERS
# ------------------------------------------
BASE_SIGMA_XY = 1#1.0  # Base error on X/Y axis (Gaussian)
# BASE_SIGMA_Z = 2#2.0   # Base error on Depth (Z) axis (Gaussian)
K_NEIGHBORS = 0#300    # How many points to average over (Controls how large the 'waves' are)
# SIGMA_AOI removed (using exact paper formula for AoI)
NOISE_CUTOFF_MM = 5.0


In [25]:
# ==========================================
# 2. LOAD CAD & COMPUTE AOI
# ==========================================
print(f"Loading Simulated CAD for Viewpoint {TARGET_VIEWPOINT}...")

sim_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{TARGET_VIEWPOINT}.pcd")
pose_path = os.path.join(SIM_DIR, f"viewpoint_pose_{TARGET_VIEWPOINT}.npy")

if not os.path.exists(sim_path):
    print(f"File not found: {sim_path}")
else:
    pcd_ideal = o3d.io.read_point_cloud(sim_path)
    
    # Ensure normals exist
    if not pcd_ideal.has_normals():
        pcd_ideal.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=10.0, max_nn=30))
        pcd_ideal.orient_normals_towards_camera_location(camera_location=np.array([0., 0., 0.]))
        # Flip normals to point OUT of surface
        normals = np.asarray(pcd_ideal.normals)
        pcd_ideal.normals = o3d.utility.Vector3dVector(-normals)
        
    # Transform CAD into Camera Space
    T_cam_to_obj = np.load(pose_path)
    T_obj_to_cam = np.linalg.inv(T_cam_to_obj)
    pcd_ideal.transform(T_obj_to_cam)
    
    print(f"Loaded and transformed {len(pcd_ideal.points)} points into Camera Space.")


Loading Simulated CAD for Viewpoint 17...
Loaded and transformed 18587 points into Camera Space.


In [26]:
# ==========================================
# 3. APPLY MANUAL DROPOUT & WAVY FIRMWARE NOISE
# ==========================================
import math
import random
points = np.asarray(pcd_ideal.points)
normals = np.asarray(pcd_ideal.normals)

print("Calculating AoI & Z-Depth...")
camera_dir = -points
camera_dir_norms = np.linalg.norm(camera_dir, axis=1, keepdims=True)
camera_dir_norm = camera_dir / camera_dir_norms

normal_norms = np.linalg.norm(normals, axis=1, keepdims=True)
normal_norm_vecs = normals / normal_norms

cos_thetas = np.clip(np.sum(camera_dir_norm * normal_norm_vecs, axis=1), -1.0, 1.0)
aois = np.degrees(np.arccos(cos_thetas))
aois = np.where(aois > 90, 180 - aois, aois)

# 1. Convert Z to meters for the paper's formula
z_m = np.abs(points[:, 2]) / 1000.0

# 2. Calculate AoI explosion multiplier using tangent asymptote
aoi_multiplier = np.zeros_like(aois)
valid_aoi = aois < 90.0
# scaled_aois removed for new formula
aois_rad = np.radians(aois[valid_aoi])
aoi_multiplier[valid_aoi] = aois_rad / ((np.pi/2.0 - aois_rad) ** 2)
aoi_multiplier[~valid_aoi] = np.inf

# 3. Calculate dynamic sigma_z (in meters)
sigma_z_m = 0.001063 + 0.0007278*z_m + 0.003949*(z_m**2) + (0.022 * (z_m**1.5) * aoi_multiplier)

# Convert back to millimeters
sigma_z_mm = sigma_z_m * 1000.0

print(f"Generating Raw TV-Static Gaussian Noise...")
raw_noise_x = np.random.normal(0, BASE_SIGMA_XY, size=len(points)) # np.zeros(len(points))
raw_noise_y = np.random.normal(0, BASE_SIGMA_XY, size=len(points)) # np.zeros(len(points))
raw_noise_z = np.random.normal(0, sigma_z_mm)
raw_noise = np.column_stack((raw_noise_x, raw_noise_y, raw_noise_z))

print("Dropping points based on max noise threshold...")
surviving_indices = np.where(np.abs(raw_noise_z) <= NOISE_CUTOFF_MM)[0]
dropped_count = len(points) - len(surviving_indices)
print(f"Dropout Complete: {dropped_count} points deleted.")

surviving_points = points[surviving_indices]
raw_noise = raw_noise[surviving_indices]

smoothed_noise = np.zeros_like(raw_noise)

if K_NEIGHBORS <= 0:
    print("K_NEIGHBORS is 0. Bypassing Firmware Blur (Using Raw Static Noise)...")
    smoothed_noise = raw_noise
else:
    print("Applying Spatial Moving Average (Simulating Firmware Blur for Wavy Effect)...")
    if len(surviving_points) > 0:
        pcd_surviving = o3d.geometry.PointCloud()
        pcd_surviving.points = o3d.utility.Vector3dVector(surviving_points)
        kdtree = o3d.geometry.KDTreeFlann(pcd_surviving)
        
        for i in range(len(surviving_points)):
            [k, idx, _] = kdtree.search_knn_vector_3d(surviving_points[i], K_NEIGHBORS)
            avg_noise = np.mean(raw_noise[idx], axis=0)
            smoothed_noise[i] = avg_noise * np.sqrt(k)

noisy_points = surviving_points + smoothed_noise

pcd_simulated_noise = o3d.geometry.PointCloud()
pcd_simulated_noise.points = o3d.utility.Vector3dVector(noisy_points)
print(f"Simulation successful! {len(surviving_points)} wavy points generated.")


Calculating AoI & Z-Depth...
Generating Raw TV-Static Gaussian Noise...
Dropping points based on max noise threshold...
Dropout Complete: 13239 points deleted.
K_NEIGHBORS is 0. Bypassing Firmware Blur (Using Raw Static Noise)...
Simulation successful! 5348 wavy points generated.


In [27]:
# ==========================================
# 4. VISUALIZATION
# ==========================================
pcd_ideal.paint_uniform_color([1, 0, 0])             # Red: Ideal CAD
pcd_simulated_noise.paint_uniform_color([0, 1, 0])   # Green: Mathematical Noise

# Translate them apart horizontally by 50mm
pcd_ideal_viz = copy.deepcopy(pcd_ideal).translate([-50, 0, 0])
pcd_noise_viz = copy.deepcopy(pcd_simulated_noise).translate([50, 0, 0])
world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50, origin=[0, 0, 0])

viz_list = [pcd_ideal_viz, pcd_noise_viz]
print("Red (Left): Ideal CAD | Green (Right): Simulated (Dropout + Wavy Noise)")
print("Close Open3D window to continue.")

o3d.visualization.draw_geometries(viz_list, window_name=f"Physics Simulation - View {TARGET_VIEWPOINT}")


Red (Left): Ideal CAD | Green (Right): Simulated (Dropout + Wavy Noise)
Close Open3D window to continue.


In [33]:
# ==========================================
# 4.5 3-WAY VISUALIZATION (INCLUDING ACTUAL SCAN)
# ==========================================
# Set up the paths to your real processed sensor data here:

# EXPERIMENT = "test_6_noise"  # Update this if your real data is in a different folder!
# PROCESSED_DIR = f"processed_data/{EXPERIMENT}"
# FULL_TRANSFORM_SAVE_DIR = f'evaluation_result/{EXPERIMENT}/merge_full_transformation.npy'

EXPERIMENT = "test_5_090angle"  # Update this if your real data is in a different folder!
PROCESSED_DIR = f"processed_data/{EXPERIMENT}"
FULL_TRANSFORM_SAVE_DIR = f'evaluation_result/{EXPERIMENT}/merge_full_transformation.npy'

real_path = os.path.join(PROCESSED_DIR, f"viewpoint_simulated_{TARGET_VIEWPOINT}.pcd")

if os.path.exists(real_path) and os.path.exists(FULL_TRANSFORM_SAVE_DIR):
    pcd_real = o3d.io.read_point_cloud(real_path)
    merge_full_transformation = np.load(FULL_TRANSFORM_SAVE_DIR)
    
    # Move real scan from Base -> CAD Space -> Camera Space
    T_target_to_object = np.linalg.inv(merge_full_transformation)
    pcd_real.transform(T_target_to_object)
    pcd_real.transform(T_obj_to_cam)
    
    pcd_ideal.paint_uniform_color([1, 0, 0])         # Red: Ideal CAD
    pcd_simulated_noise.paint_uniform_color([0, 1, 0]) # Green: Mathematical Noise
    pcd_real.paint_uniform_color([0, 0.2, 0.8])      # Blue: Real Scan
    
    # Translate them apart horizontally by 100mm
    pcd_ideal_viz = copy.deepcopy(pcd_ideal).translate([-110, -10, -35])
    pcd_noise_viz = copy.deepcopy(pcd_simulated_noise).translate([0, 0, 0])
    pcd_real_viz = copy.deepcopy(pcd_real).translate([110, 10, 35])
    pcd_noise_viz.estimate_normals()
    pcd_real_viz.estimate_normals()
    
    viz_list = [pcd_ideal_viz, pcd_noise_viz, pcd_real_viz]
    print("Red (Left): Ideal CAD | Green (Center): Simulated | Blue (Right): Real Sensor")
    print("Close Open3D window to continue.")
    
    o3d.visualization.draw_geometries(viz_list, window_name=f"3-Way Physics Comparison - View {TARGET_VIEWPOINT}")
else:
    print(f"Could not find real scan data at: {real_path}")
    print(f"Or missing transformation matrix at: {FULL_TRANSFORM_SAVE_DIR}")


Red (Left): Ideal CAD | Green (Center): Simulated | Blue (Right): Real Sensor
Close Open3D window to continue.


In [23]:
# ==========================================
# 6. BATCH PROCESS ALL VIEWPOINTS
# ==========================================
import os
import glob
import math
import numpy as np
import open3d as o3d

# CONFIGURATION
# WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV"]
WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV", "TH0041AV", "TH0042AV", "TH0051AV", "TH0052AV", "TH0061AV", "TH0062AV", "TH0071AV", "TH0072AV"]

# SIGMA_AOI removed
BASE_SIGMA_XY = 1.0  
# BASE_SIGMA_Z = 2.0   
K_NEIGHBORS = 300
NOISE_CUTOFF_MM = 5.0

for workpiece in WORKPIECES:
    print(f"\n==========================================")
    print(f"Processing Workpiece: {workpiece}")
    print(f"==========================================")
    
    SIM_DIR = f"viewpoints_candidate/testing_data/test_9_simulation_3/{workpiece}"
    OUT_DIR = f"simulation/test_9_simulation_3/{workpiece}"
    
    if not os.path.exists(OUT_DIR):
        os.makedirs(OUT_DIR)
        
    sim_files = glob.glob(os.path.join(SIM_DIR, "viewpoint_simulated_*.pcd"))
    print(f"Found {len(sim_files)} viewpoints for {workpiece}.")
    
    for sim_path in sim_files:
        basename = os.path.basename(sim_path)
        if "noise" in basename:
            continue
        view_idx_str = basename.replace("viewpoint_simulated_", "").replace(".pcd", "")
        view_idx = int(view_idx_str)
        
        pose_path = os.path.join(SIM_DIR, f"viewpoint_pose_{view_idx}.npy")
        if not os.path.exists(pose_path):
            continue
            
        # 1. LOAD & TRANSFORM TO CAMERA SPACE
        pcd_ideal = o3d.io.read_point_cloud(sim_path)
        
        if not pcd_ideal.has_normals():
            pcd_ideal.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=10.0, max_nn=30))
            pcd_ideal.orient_normals_towards_camera_location(camera_location=np.array([0., 0., 0.]))
            normals = np.asarray(pcd_ideal.normals)
            pcd_ideal.normals = o3d.utility.Vector3dVector(-normals)
            
        T_cam_to_obj = np.load(pose_path)
        T_obj_to_cam = np.linalg.inv(T_cam_to_obj)
        pcd_ideal.transform(T_obj_to_cam)
        
        points = np.asarray(pcd_ideal.points)
        normals = np.asarray(pcd_ideal.normals)
        
        # 2. CALCULATE AOI & Z-Depth
        camera_dir = -points
        camera_dir_norms = np.linalg.norm(camera_dir, axis=1, keepdims=True)
        camera_dir_norm = camera_dir / camera_dir_norms
        normal_norms = np.linalg.norm(normals, axis=1, keepdims=True)
        normal_norm_vecs = normals / normal_norms
        
        cos_thetas = np.clip(np.sum(camera_dir_norm * normal_norm_vecs, axis=1), -1.0, 1.0)
        aois = np.degrees(np.arccos(cos_thetas))
        aois = np.where(aois > 90, 180 - aois, aois)
        
        # Convert Z to meters
        z_m = np.abs(points[:, 2]) / 1000.0
        
        # Calculate AoI explosion multiplier
        aoi_multiplier = np.zeros_like(aois)
        valid_aoi = aois < 90.0
        aois_rad = np.radians(aois[valid_aoi])
        aoi_multiplier[valid_aoi] = aois_rad / ((np.pi/2.0 - aois_rad) ** 2)
        aoi_multiplier[~valid_aoi] = np.inf
        
        # Calculate dynamic sigma_z (in meters)
        sigma_z_m = 0.001063 + 0.0007278*z_m + 0.003949*(z_m**2) + (0.022 * (z_m**1.5) * aoi_multiplier)
        sigma_z_mm = sigma_z_m * 1000.0
        
        # 3. SPATIAL WAVY NOISE & DROPOUT
        raw_noise_x = np.random.normal(0, BASE_SIGMA_XY, size=len(points)) # np.zeros(len(points))
        raw_noise_y = np.random.normal(0, BASE_SIGMA_XY, size=len(points)) # np.zeros(len(points))
        raw_noise_z = np.random.normal(0, sigma_z_mm)
        raw_noise = np.column_stack((raw_noise_x, raw_noise_y, raw_noise_z))
        
        surviving_indices = np.where(np.abs(raw_noise_z) <= NOISE_CUTOFF_MM)[0]
        
        surviving_points = points[surviving_indices]
        raw_noise = raw_noise[surviving_indices]
        
        smoothed_noise = np.zeros_like(raw_noise)
        
        if len(surviving_points) > 0:
            pcd_surviving = o3d.geometry.PointCloud()
            pcd_surviving.points = o3d.utility.Vector3dVector(surviving_points)
            kdtree = o3d.geometry.KDTreeFlann(pcd_surviving)
            
            for i in range(len(surviving_points)):
                [k, idx, _] = kdtree.search_knn_vector_3d(surviving_points[i], K_NEIGHBORS)
                avg_noise = np.mean(raw_noise[idx], axis=0)
                smoothed_noise[i] = avg_noise * np.sqrt(k)
            
        noisy_points = surviving_points + smoothed_noise
        
        # 5. TRANSFORM BACK & SAVE
        pcd_simulated_noise = o3d.geometry.PointCloud()
        pcd_simulated_noise.points = o3d.utility.Vector3dVector(noisy_points)
        
        pcd_simulated_noise.transform(T_cam_to_obj)
        
        out_path = os.path.join(OUT_DIR, f"viewpoint_simulated_noise_{view_idx}.pcd")
        o3d.io.write_point_cloud(out_path, pcd_simulated_noise)

print(f"\nALL BATCHES DONE!")



Processing Workpiece: TH0011AV
Found 432 viewpoints for TH0011AV.

Processing Workpiece: TH0012AV
Found 432 viewpoints for TH0012AV.

Processing Workpiece: TH0021AV
Found 432 viewpoints for TH0021AV.

Processing Workpiece: TH0022AV
Found 432 viewpoints for TH0022AV.

Processing Workpiece: TH0031AV
Found 432 viewpoints for TH0031AV.

Processing Workpiece: TH0032AV
Found 432 viewpoints for TH0032AV.

Processing Workpiece: TH0041AV
Found 432 viewpoints for TH0041AV.

Processing Workpiece: TH0042AV
Found 432 viewpoints for TH0042AV.

Processing Workpiece: TH0051AV
Found 432 viewpoints for TH0051AV.

Processing Workpiece: TH0052AV
Found 432 viewpoints for TH0052AV.

Processing Workpiece: TH0061AV
Found 432 viewpoints for TH0061AV.

Processing Workpiece: TH0062AV
Found 432 viewpoints for TH0062AV.

Processing Workpiece: TH0071AV
Found 432 viewpoints for TH0071AV.

Processing Workpiece: TH0072AV
Found 432 viewpoints for TH0072AV.

ALL BATCHES DONE!


In [10]:
# ==========================================
# 7. COMBINE AND AVERAGE NOISY VIEWPOINTS
# ==========================================
import os
import glob
import open3d as o3d

VOXEL_SIZE = 1.5  # mm - the size of the voxel to "average" points together

# WORKPIECES = ['TH0011AV']
WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV"]


for workpiece in WORKPIECES:
    sim_dir = f"simulation/{EXPERIMENT}/{workpiece}"
    
    if not os.path.exists(sim_dir):
        print(f"Skipping {workpiece}, directory not found: {sim_dir}")
        continue
        
    print(f"==========================================")
    print(f"Processing Workpiece: {workpiece}")
    
    # 1. Find all noisy simulated viewpoints
    search_pattern = os.path.join(sim_dir, "viewpoint_simulated_noise_*.pcd")
    noisy_files = glob.glob(search_pattern)
    
    if not noisy_files:
        print(f"No noisy PCDs found for {workpiece}.")
        continue
        
    print(f"Found {len(noisy_files)} noisy viewpoints. Combining...")
    
    # 2. Combine all points
    combined_pcd = o3d.geometry.PointCloud()
    for pcd_file in noisy_files:
        pcd = o3d.io.read_point_cloud(pcd_file)
        combined_pcd += pcd
        
    print(f"Total points before averaging: {len(combined_pcd.points)}")
    
    # 3. Average them by Voxel Downsampling
    averaged_pcd = combined_pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)
    
    print(f"Total points after averaging: {len(averaged_pcd.points)}")
    
    # 4. Save the result
    out_path = os.path.join(sim_dir, "viewpoint_simulated_noise_averaged.pcd")
    o3d.io.write_point_cloud(out_path, averaged_pcd)
    print(f"Saved averaged point cloud to: {out_path}\n")
    
    # 5. Visualize the result
    print(f"Opening visualization for {workpiece}...")
    averaged_pcd.paint_uniform_color([0.2, 0.8, 0.2]) # Green for averaged
    averaged_pcd.estimate_normals()
    o3d.visualization.draw_geometries(
        [averaged_pcd],
        window_name=f"Noisy Averaged PCD - {workpiece}",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Processing Workpiece: TH0011AV
Found 433 noisy viewpoints. Combining...
Total points before averaging: 2468497
Total points after averaging: 59102
Saved averaged point cloud to: simulation/test_9_simulation_3/TH0011AV\viewpoint_simulated_noise_averaged.pcd

Opening visualization for TH0011AV...
Processing Workpiece: TH0012AV
Found 432 noisy viewpoints. Combining...
Total points before averaging: 2088957
Total points after averaging: 64542
Saved averaged point cloud to: simulation/test_9_simulation_3/TH0012AV\viewpoint_simulated_noise_averaged.pcd

Opening visualization for TH0012AV...
Processing Workpiece: TH0021AV
Found 432 noisy viewpoints. Combining...
Total points before averaging: 2507026
Total points after averaging: 56695
Saved averaged point cloud to: simulation/test_9_simulation_3/TH0021AV\viewpoint_simulated_noise_averaged.pcd

Opening visualization for TH0021AV...
Processing Workpiece: TH0022AV
Found 216 noisy viewpoints. Combining...
Total points before averaging: 1303397
T

In [8]:
averaged_pcd

PointCloud with 27712 points.